# SpecDist — Colab T4 Quickstart

**Distil a Qwen3-0.6B draft from a Qwen3 teacher on a free T4.**

| CONFIG | Teacher | Steps | Time | Research-valid? |
|--------|---------|-------|------|-----------------|
| `kaggle` | **8B (4-bit NF4)** | 1000 | ~5-8 h | **YES** — same 8B teacher as the A100 paper run; loss rankings transfer |
| `colab` | 4B (BF16) | 500 | ~2-4 h | No — 4B ≠ paper 8B; dev / quota-fallback only |
| `colab_lite` | 1.7B (BF16) | 300 | ~25 min | No — crash-check only |

> **Config names describe the HARDWARE+teacher, not the platform.**
> `kaggle` = "T4 + 8B-NF4" and runs on **any** T4 — Colab free, Kaggle, or Modal.
> It is the research-valid setting on a free T4. Use it here on Colab.
> Do **not** run the 8B teacher with `colab` (4B/BF16) — 8B in BF16 (~16 GB)
> OOMs a 15 GB T4 and silently falls back to CPU. A startup guard now blocks this.

Set `CONFIG` in Cell 0 and **Runtime → Run all** (Ctrl+F9).  
For A100 (8B teacher BF16, 2000 steps, full GSM8K) use `a100_quickstart.ipynb`.
For the research-valid 8B run on a free T4, set `CONFIG = "kaggle"` below.

| Cell | What it does | Run time |
|------|-------------|----------|
| 0. Bootstrap | **One-shot: setup + auth + run** | ~3 min to start |
| 1. Setup | Mount Drive, clone repo, install deps | ~3 min |
| 2. Auth | W&B + HuggingFace | ~30 s |
| 3. Run | Full pipeline (train → merge → eval) | ~25 min / ~4 h |
| 4. Resume | After session death — skips completed steps | ~1 min + rest |
| 5. Monitor | State + log tail (auto-refresh option) | instant |
| 6. Dashboard | Live results dashboard in a browser tab | ~10 s |
| 7. Tree loss (single) | Train one tree loss | ~20-40 min |
| 8. Tree losses (all) | Train all 9 tree losses sequentially | ~4-5 h |

### Prerequisites
1. **GPU runtime**: Runtime → Change runtime type → **T4 GPU**
2. **Colab Secrets** (left sidebar → 🔑 icon):
   - `GITHUB_TOKEN` — **required** (private repo).  
     Create a classic PAT with `repo` scope at https://github.com/settings/tokens
   - `WANDB_API_KEY` — from https://wandb.ai/authorize
   - `HF_TOKEN` — from https://huggingface.co/settings/tokens
3. Google Drive (mounted automatically by Cell 0 or Cell 1)

### Session timeouts
Free T4 sessions cap at ~12 h. Cell 0 injects a 45 s JS keep-alive.  
If the session dies mid-run, re-run Cell 0 (or Cell 4) — the pipeline  
resumes from the last checkpoint automatically.

In [ ]:
# =============================================================================
# Cell 0 — ONE-SHOT BOOTSTRAP
# Edit CONFIG + flags below, then Runtime → Run all (Ctrl+F9).
# All training settings (teacher model, steps, lr) come from the YAML profile
# selected by CONFIG — nothing is hard-coded here.
#
# CONFIG options (all run on a free Colab T4):
#   kaggle      — 8B teacher 4-bit NF4, 1000 steps  — RESEARCH-VALID (same 8B
#                 as the A100 paper run; rankings transfer). Name is T4-generic,
#                 NOT Kaggle-only — it means 'T4 + 8B-NF4' and works here on Colab.
#   colab       — 4B teacher BF16, 500 steps  — dev/fallback only (4B ≠ paper 8B)
#   colab_lite  — 1.7B teacher BF16, 300 steps — crash-check only
# =============================================================================

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"

# ── Edit these ────────────────────────────────────────────────────────────────────────────
CONFIG      = "kaggle"       # kaggle (8B NF4, research-valid) | colab (4B) | colab_lite (1.7B)
SMOKE       = False          # True = 10-step crash check (~5 min)
BACKGROUND  = False          # True = background; monitor with Cell 5
LOSSES      = None           # None = all losses  |  "kl,ebe" = subset
EXTRA_ARGS  = []             # e.g. ["--seed_override", "123"]
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

def _prereq_token(name):
    """Read a Colab secret before deploy_utils is available (needed for git clone)."""
    try:
        from google.colab import userdata; v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh
else: print("\u26a0 GITHUB_TOKEN not set \u2014 clone will fail for a private repo.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Repo updated")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, DRIVE_ROOT, REPO_DIR, gbv_dir=GBV_DIR, mount_drive=True)
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR,
             smoke=SMOKE, losses=LOSSES,
             background=BACKGROUND, extra_args=EXTRA_ARGS)

In [ ]:
# =============================================================================
# Cell 1 — Setup  (alternative to Cell 0: run Cells 1 → 2 → 3 separately)
# Mount Drive, clone repo, install deps.
# =============================================================================
import os, subprocess, sys

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)
print(f"✓ Artifacts persist at: {DRIVE_ROOT}")

# Clone (GITHUB_TOKEN must be in Colab Secrets — private repo)
# Minimal bootstrap: deploy_utils (which has _auth_repo_url) needs the clone first.
try: from google.colab import userdata; gh = userdata.get("GITHUB_TOKEN") or ""
except Exception: gh = os.environ.get("GITHUB_TOKEN", "")
if not gh:
    print("⚠ GITHUB_TOKEN not set — clone will fail for a private repo.")
clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise subprocess.CalledProcessError(r.returncode, r.args)
    print("✓ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✓ Repo updated")

GBV_DIR = f"{REPO_DIR}/gbv-research"
os.chdir(GBV_DIR)

# Install deps (delegates to shared utils)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import install_deps, fetch_training_data
install_deps(GBV_DIR)
fetch_training_data(GBV_DIR)
print("--- Setup done. Run Cell 2. ---")

In [ ]:
# =============================================================================
# Cell 2 — Authenticate  (W&B + HuggingFace + GPU check)
# Reads WANDB_API_KEY and HF_TOKEN from Colab Secrets.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import auth_wandb, auth_hf, check_gpu

auth_wandb()
auth_hf()
check_gpu()
print("--- Auth done. Run Cell 3. ---")

In [ ]:
# =============================================================================
# Cell 3 — Run pipeline  (flat losses: kl, jsd, l1, ebe, …)
# Prerequisite: Cells 1 + 2 (or Cell 0).
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, fetch_training_data

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ────────────────────────────────────────────────────────────────
CONFIG      = "kaggle"   # kaggle (8B NF4, research-valid) | colab (4B) | colab_lite (1.7B)
SMOKE       = False      # True = 10-step crash check
BACKGROUND  = False      # True = background process; monitor with Cell 5
LOSSES      = None       # None = all  |  "kl,ebe" = subset
# ─────────────────────────────────────────────────────────────────────────────

fetch_training_data(GBV_DIR)
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSSES, background=BACKGROUND)

In [ ]:
# =============================================================================
# Cell 4 — Resume after session death
# Re-mounts Drive, re-clones if needed, re-auths, resumes pipeline.
# The pipeline skips already-completed steps automatically.
# =============================================================================

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
REPO_URL   = "https://github.com/Rmuk655/Distill-Spec-Research.git"
REPO_DIR   = "/content/Distill-Spec-Research"
GBV_DIR    = f"{REPO_DIR}/gbv-research"
CONFIG     = "colab"   # must match what was used in Cell 3

import os, subprocess, sys

def _prereq_token(name):
    """Read a Colab secret before deploy_utils is available (needed for git clone)."""
    try:
        from google.colab import userdata; v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, "")

gh = _prereq_token("GITHUB_TOKEN")
if gh: os.environ["GITHUB_TOKEN"] = gh
else: print("\u26a0 GITHUB_TOKEN not set \u2014 clone will fail for a private repo.")

clone_url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr.strip()); raise subprocess.CalledProcessError(r.returncode, r.args)
    print("Repo cloned")
else:
    if gh: subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", clone_url], capture_output=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Repo updated")

sys.modules.pop("deploy_utils", None)
sys.path.insert(0, f"{GBV_DIR}/deploy")
from deploy_utils import bootstrap, run_pipeline

bootstrap(CONFIG, DRIVE_ROOT, REPO_DIR, gbv_dir=GBV_DIR, mount_drive=True)
print(f"\nResuming {CONFIG} — completed steps are skipped automatically.\n")
_ = run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, background=True)

In [ ]:
# =============================================================================
# Cell 6 — Monitor  (safe to run any time, including while pipeline runs)
# AUTO_REFRESH = True → live tail; interrupt cell to stop.
#
# CONFIG must match the PROFILE used in Cell 7 or Cell 8.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import monitor

DRIVE_ROOT   = "/content/drive/MyDrive/specdist"
# ── Must match Cell 7/8 PROFILE ─────────────────────────────────────────────
CONFIG       = "profiles/colab_lite_tree_losses"
# Other options:
#   "profiles/colab_tree_losses"      # 4B teacher (colab_quickstart default)
#   "colab"                           # legacy flat-loss runs
#   "colab_lite"                      # legacy 1.7B flat-loss runs
# ────────────────────────────────────────────────────────────────────────────
AUTO_REFRESH = False     # True = live tail loop (interrupt cell to stop)
REFRESH_SECS = 20

monitor(DRIVE_ROOT, CONFIG, auto_refresh=AUTO_REFRESH, refresh_secs=REFRESH_SECS)

In [ ]:
# =============================================================================
# Cell 6 — Dashboard  (safe to run while training)
# Opens a live results dashboard in a new browser tab.
# Stop: Runtime → Interrupt execution.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import start_dashboard

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

start_dashboard(DRIVE_ROOT, GBV_DIR)

---

## Tree-Structured Losses (Track B) — T4

Use **Cell 7** for one loss at a time, **Cell 8** for all 9.  
All training settings come from the YAML profile — see `show_profile()` output.

| PROFILE | Teacher | Steps | Time/loss |
|---------|---------|-------|-----------|
| `profiles/colab_lite_tree_losses` | 1.7B | 300 | ~10-15 min |
| `profiles/colab_tree_losses` | 4B | 500 | ~20-40 min |

| Loss | Targets | Notes |
|------|---------|-------|
| `kl_tree` | Forward KL at each tree node | H5 primary baseline |
| `rev_kl_tree` | Reverse KL (mode-seeking) | |
| `jsd_tree` | Symmetric JSD | Bounded [0, log 2] |
| `bv_tree` | BV acceptance integral | Targets `bv_verify` directly |
| `gbv_tree` | GBV + q-skew | Stable at K≤4 |
| `traversal_tree` | Traversal leaf-weight product | Targets `traversal_verify` |
| `ebe_tree` | On-policy EBE ablation | |
| `online_kl_tree` | Online tree KL — no replay buffer | H6 test |
| `online_ebe_tree` | Online tree EBE | |

**Prerequisite:** Cells 1 + 2 must have run (or Cell 0).

In [ ]:
# =============================================================================
# Cell 7 — Train a single tree loss (T4)
# Change LOSS and re-run for each one. Completed steps are always skipped.
# background=True: cell returns immediately — run Cell 6 (monitor) in parallel.
# =============================================================================
import sys
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, show_profile, keep_alive

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ──────────────────────────────────────────────────────────────────────────────
# Offline:  kl_tree | rev_kl_tree | jsd_tree | bv_tree | gbv_tree
#           traversal_tree | ebe_tree
# Online:   online_kl_tree | online_ebe_tree
LOSS    = "kl_tree"
PROFILE = "profiles/colab_tree_losses"   # or colab_lite_tree_losses (1.7B, ~10 min)
SMOKE   = False
# ─────────────────────────────────────────────────────────────────────────────────

import torch
if torch.cuda.is_available() and torch.cuda.mem_get_info(0)[1] / 1024**3 > 30:
    print("NOTE: A100 detected — consider profiles/a100_tree_losses (8B, 2000 steps)")

keep_alive()
show_profile(PROFILE, GBV_DIR)
print(f"  Loss: {LOSS}")

# background=True: log-tailing thread streams output; run Cell 6 (monitor) anytime.
run_pipeline(PROFILE, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSS, background=True)

In [ ]:
# =============================================================================
# Cell 8 — Train ALL tree losses sequentially (T4)
# Re-run to resume — completed steps are always skipped.
# Runs in a background thread — Cell 6 (monitor) and dashboard stay available.
# =============================================================================
import sys, threading
sys.path.insert(0, "/content/Distill-Spec-Research/gbv-research/deploy")
from deploy_utils import run_pipeline, show_profile, keep_alive

DRIVE_ROOT = "/content/drive/MyDrive/specdist"
GBV_DIR    = "/content/Distill-Spec-Research/gbv-research"

# ── Edit these ──────────────────────────────────────────────────────────────────────────────
PROFILE = "profiles/colab_tree_losses"  # or colab_lite_tree_losses (~1.5-2 h total)
LOSSES_TO_RUN = [
    "kl_tree", "rev_kl_tree", "jsd_tree",        # universal divergences
    "bv_tree", "gbv_tree", "traversal_tree",      # verifier-specific
    "ebe_tree",                                    # off-policy ablation
    "online_kl_tree", "online_ebe_tree",           # H6: online tree vs flat online
]
SMOKE = False
# ─────────────────────────────────────────────────────────────────────────────────

keep_alive()
show_profile(PROFILE, GBV_DIR)
print(f"  Losses: {len(LOSSES_TO_RUN)} × tree  |  resume-safe\n")

def _run_all_losses():
    for i, loss in enumerate(LOSSES_TO_RUN):
        print(f"{'='*50}\n[{i+1}/{len(LOSSES_TO_RUN)}] {loss}\n{'='*50}")
        # background=True streams log output; proc.wait() keeps losses sequential
        # without blocking the kernel — Cell 6 (monitor) can run in parallel.
        proc = run_pipeline(PROFILE, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=loss,
                            background=True)
        proc.wait()
        rc = proc.returncode
        if rc != 0:
            print(f"  [FAIL] {loss} exited {rc} — re-run to resume.")
            break
    else:
        print("\n✓ All tree losses complete — run Cell 6 for the dashboard.")

threading.Thread(target=_run_all_losses, daemon=True).start()
print("Losses running in background — Cell 6 (monitor) and dashboard are available now.")

In [ ]:
# =============================================================================
# Cell 4 — Train ONE loss  (Colab — one loss at a time, --skip_existing handles resume)
# Change LOSS below and re-run.  --skip_existing skips completed steps.
# This is the standard per-session workflow on Kaggle:
#   Session 1: LOSS = 'kl'        Session 2: LOSS = 'rev_kl' etc.
# On persistent machines (A100): change LOSS each time you want the next loss.
# =============================================================================
import sys
sys.path.insert(0, '/content/Distill-Spec-Research/gbv-research/deploy')
from deploy_utils import run_pipeline

DRIVE_ROOT = '/content/drive/MyDrive/specdist'
GBV_DIR    = '/content/Distill-Spec-Research/gbv-research'

# ── Edit this each session ───────────────────────────────────────────────────
LOSS   = 'kl'      # kl | rev_kl | jsd | l1 | ebe | ebe_single
#                   # kl_tree | bv_tree | gbv_tree | traversal_tree
#                   # rev_kl_tree | jsd_tree | naive_tree | nss_tree
#                   # specinfer_tree | spectr_tree | khisti_tree
CONFIG = 'colab'   # must match the CONFIG used in Cell 0
SMOKE  = False         # True = 10-step crash check
# ─────────────────────────────────────────────────────────────────────────────

# Does: baseline eval (once, --skip_existing) → train LOSS → merge → eval GSM8K
run_pipeline(CONFIG, DRIVE_ROOT, GBV_DIR, smoke=SMOKE, losses=LOSS, background=False)


In [ ]:
# =============================================================================
# Cell 5 — Eval ONE trained model  (no retraining)
# Use when the checkpoint is already done but you want to re-run eval
# (different K, different n_prompts, extra verifier modes, etc.)
# Runs evaluate.py directly — bypasses the full pipeline orchestration.
# =============================================================================
import sys, os
sys.path.insert(0, '/content/Distill-Spec-Research/gbv-research')
GBV_DIR    = '/content/Distill-Spec-Research/gbv-research'
DRIVE_ROOT = '/content/drive/MyDrive/specdist'

# ── Edit these ───────────────────────────────────────────────────────────────
LOSS    = 'kl'           # which trained model to evaluate
CONFIG  = 'colab'     # must match training config
MODES   = 'alpha,bv,gbv,traversal,specinfer,naive'  # verifier modes
K       = '3'            # draft paths
TEMP    = '1.0'          # sampling temperature
N       = '100'          # number of eval prompts
DATASET = 'gsm8k'        # gsm8k | humaneval | math500 | mtbench | alpaca
# ─────────────────────────────────────────────────────────────────────────────

import subprocess
ckpt = os.path.join(DRIVE_ROOT, 'checkpoints', f'{LOSS.replace("_","_")}-gsm8k_merged')
if not os.path.isdir(ckpt):
    ckpt = os.path.join(DRIVE_ROOT, 'checkpoints', f'{LOSS}-gsm8k_merged')
    print(f'Trying: {ckpt}')
if not os.path.isdir(ckpt):
    raise FileNotFoundError(f'Merged checkpoint not found: {ckpt}\n'
                            'Run Cell 4 first to train + merge the model.')

# Import the target model path from the running config
import yaml
cfg_path = os.path.join(GBV_DIR, 'orchestration', 'configs', f'{CONFIG}.yaml')
target = yaml.safe_load(open(cfg_path))['models']['target']

cmd = [
    'python', os.path.join(GBV_DIR, 'orchestration', 'evaluate.py'),
    '--student', ckpt,
    '--teacher', target,
    '--student_label', LOSS,
    '--datasets', DATASET,
    '--modes', MODES,
    '--K', K, '--temperature', TEMP, '--n', N,
    '--skip_fetch', '--skip_existing',
    '--storage_root', DRIVE_ROOT,
    '--loss_name', LOSS,
]
print('Running:', ' '.join(cmd[-10:]))
result = subprocess.run(cmd, cwd=GBV_DIR)
print('Done — refresh dashboard to see results.')
